# Features to use — downstream features affected by the aggregator fix

**Why this notebook exists.** The payment-pattern aggregator denominator fix changes the trended
delinquency-rate features (`trade_..._percent_of_DQ<n>_in_last_<m>_months__...`). The end goal is to
measure how much that fix actually moves **model performance** — by training **two models on the
exact same feature set**, differing only in the aggregator that produced the data:

- **NEW** — features built from `processed_new/` (fixed aggregator)
- **OLD** — features built from `processed_old/` (original aggregator)

Same feature columns, same applicants → any performance difference is attributable to the fix alone.

**This notebook's job** is step 1: identify that shared, downstream feature set and **freeze it** to
`payment_processing_research_data/Features_To_Use.json`, so both the NEW and OLD model runs read the
identical list.

## 1. Load one processed table just to read its column names

In [10]:
import os, sys, json
import pandas as pd


import configs
from configs import DATA_DIR

# Any processed slice has the same columns; one part file is enough for the schema.
example_path = os.path.join(DATA_DIR, 'transunion', 'test', 'processed_new', 'part-000.parquet')
example_processed = pd.read_parquet(example_path)
print(f'{example_processed.shape[1]:,} columns in {example_path}')

8,848 columns in /home/jag/payment-processor-research/payment_processing_research_data/transunion/test/processed_new/part-000.parquet


## 2. Select the downstream features the aggregator fix affects

These are the trended `percent_of_DQ..._in_last_..._months` features — the ones whose **denominator**
the fix recomputes. This is the candidate set both models will be trained on.

In [11]:
features = [
    c for c in example_processed.columns
    if 'percent_of' in c.lower()
    and 'dq'        in c.lower()
    and 'in_last'   in c.lower()
]
print(f'{len(features):,} downstream features selected')
features[:10]

1,143 downstream features selected


['trade_mean_percent_of_DQ30_or_greater_in_last_6_months__all_open_accounts',
 'trade_min_percent_of_DQ30_or_greater_in_last_6_months__all_open_accounts',
 'trade_max_percent_of_DQ30_or_greater_in_last_6_months__all_open_accounts',
 'trade_mean_percent_of_DQ30_in_last_12_months__all_open_accounts',
 'trade_min_percent_of_DQ30_in_last_12_months__all_open_accounts',
 'trade_max_percent_of_DQ30_in_last_12_months__all_open_accounts',
 'trade_mean_percent_of_DQ30_in_last_24_months__all_open_accounts',
 'trade_min_percent_of_DQ30_in_last_24_months__all_open_accounts',
 'trade_max_percent_of_DQ30_in_last_24_months__all_open_accounts',
 'trade_mean_percent_of_DQ60_in_last_12_months__all_open_accounts']

## 3. Freeze the starting feature list to `Features_To_Use.json`

In [12]:
out_path = os.path.join(DATA_DIR, 'Features_To_Use.json')
with open(out_path, 'w') as f:
    json.dump(features, f, indent=2)
print(f'wrote {len(features):,} features -> {out_path}')

wrote 1,143 features -> /home/jag/payment-processor-research/payment_processing_research_data/Features_To_Use.json


## 4. Next step (separate notebook)

Load `Features_To_Use.json` and train two models on this identical feature set — one reading the
`processed_new/` columns, one reading `processed_old/` — on the same sampled applicants, then compare
performance. The only thing that differs between the two runs is the aggregator, so the gap isolates
the impact of the fix.